In [ ]:
#@title Celula 1: Clonar ou atualizar o repositorio no Google Colab
import os
import subprocess

REPO_URL = "https://github.com/lucasrs012007-dev/gps-canal-dark.git"
REPO_DIR = "/content/gps-canal-dark"

if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
    subprocess.run(["git", "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print("Repositorio pronto em:", os.getcwd())
subprocess.run(["python3", "pipeline/historico.py", "verificar"], check=True)


In [ ]:
#@title Celula 2: Narracao (ElevenLabs via param/Secrets ou edge-tts), Sound Design e Render 9:16
# Deixe vazio para ler automaticamente dos Secrets do Colab (google.colab.userdata)
# ou usar o fallback gratuito edge-tts (pt-BR-AntonioNeural em tom grave).
ELEVENLABS_API_KEY = "" #@param {type:"string"}
ELEVENLABS_VOICE_ID = "" #@param {type:"string"}
REGERAR_AUDIO = True #@param {type:"boolean"}
MODO_TESTE = False #@param {type:"boolean"}

import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/content/gps-canal-dark")
if REPO_DIR.is_dir():
    os.chdir(REPO_DIR)

def ler_colab_secret(nome: str) -> str:
    try:
        from google.colab import userdata
        val = userdata.get(nome)
        return (val or "").strip()
    except Exception:
        return ""

api_key = ELEVENLABS_API_KEY.strip()
origem_chave = "campo #@param"
if not api_key:
    api_key = ler_colab_secret("ELEVENLABS_API_KEY")
    origem_chave = "Colab Secrets (userdata)"
if not api_key:
    api_key = os.environ.get("ELEVENLABS_API_KEY", "").strip()
    origem_chave = "ambiente/.env"

voice_id = ELEVENLABS_VOICE_ID.strip()
if not voice_id:
    voice_id = ler_colab_secret("ELEVENLABS_VOICE_ID")
if not voice_id:
    voice_id = os.environ.get("ELEVENLABS_VOICE_ID", "").strip()

if api_key:
    os.environ["ELEVENLABS_API_KEY"] = api_key
    print(f"Motor de voz selecionado: ElevenLabs (eleven_multilingual_v2) via {origem_chave}.")
else:
    os.environ.pop("ELEVENLABS_API_KEY", None)
    print("Motor de voz selecionado: edge-tts (pt-BR-AntonioNeural, tom grave).")

if voice_id:
    os.environ["ELEVENLABS_VOICE_ID"] = voice_id
else:
    os.environ.pop("ELEVENLABS_VOICE_ID", None)

# 0. Validar regra Sempre Video Novo e descobrir saida automatica (ex.: output/short_02.mp4)
subprocess.run([sys.executable, "pipeline/historico.py", "verificar"], check=True)
saida_mp4 = subprocess.run(
    [sys.executable, "pipeline/historico.py", "saida"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()
print(f"Arquivo de saida alvo: {saida_mp4}")

# 1. Garantir ffmpeg
if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)

# 2. Garantir edge-tts (usado no fallback caso ElevenLabs nao esteja configurado)
if not api_key:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "edge-tts>=7"], check=True)

# 3. Garantir Google Chrome oficial (evita o stub quebrado do snap chromium-browser)
if not shutil.which("google-chrome") and not shutil.which("chromium"):
    deb = "/tmp/google-chrome-stable_current_amd64.deb"
    subprocess.run(
        ["wget", "-q", "-O", deb, "https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb"],
        check=True,
    )
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", deb], check=True)
    Path(deb).unlink(missing_ok=True)

# 4. Gerar narracao + timestamps.json + eventos.json e sintetizar sound design (-14 LUFS)
mix_path = Path("audio/mix_final.mp3")
if REGERAR_AUDIO or not mix_path.is_file() or mix_path.stat().st_size == 0:
    print("\n[1/3] Gerando narracao, timestamps.json e eventos.json...")
    subprocess.run([sys.executable, "audio/gerar_narracao.py"], check=True)
    print("\n[2/3] Sintetizando sound design e masterizando audio/mix_final.mp3...")
    subprocess.run([sys.executable, "audio/sound_design.py"], check=True)
else:
    print("Reaproveitando audio/mix_final.mp3 existente (REGERAR_AUDIO=False).")

dur = subprocess.run(
    [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=nw=1:nk=1",
        str(mix_path),
    ],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()
print(f"Duracao real de audio/mix_final.mp3: {float(dur):.2f}s")

# 5. Renderizar MP4 vertical 1080x1920 com legendas pop-in sincronizadas
print(f"\n[3/3] Renderizando {saida_mp4}...")
cmd_render = [sys.executable, "render/renderizar.py", "--legendas"]
if MODO_TESTE:
    cmd_render.append("--teste")
subprocess.run(cmd_render, check=True)


In [ ]:
#@title Celula 3: Ouvir audio/mix_final.mp3, conferir MP4 e baixar o Short gerado
import subprocess
import sys
from pathlib import Path
from IPython.display import Audio, display
from google.colab import files

saida_mp4 = subprocess.run(
    [sys.executable, "pipeline/historico.py", "saida"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()

audio_preview = Path("audio/mix_final.mp3")
if not audio_preview.is_file():
    audio_preview = Path("audio/narracao.mp3")

print(f"Previa da mix final ({audio_preview}):")
display(Audio(str(audio_preview)))

print(f"Metadados do video ({saida_mp4}):")
subprocess.run(
    [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration:stream=codec_type,codec_name,width,height,r_frame_rate,pix_fmt",
        "-of", "default=nw=1",
        saida_mp4,
    ],
    check=True,
)

files.download(saida_mp4)
